In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor
from scipy.stats import norm

sns.set_style("white")
sns.set(font_scale = 1.5)
sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

d_abrev = {"LiangSteinNeuron":"FC exc. neur.", "FetalChondrocytes":"F chond.", "SertoliMale":"FG sertoli", "preGC_IIaFemale":"FG preGC IIa",\
          "NeuralFemale":"FG neur.", "FetalGonadImmuneFemale":"FG immune", "VIP":"AC VIP inh. neur.", "LiangSteinProgenitor":"FC prog.",\
          "AdultHeartVentricularCardiomyocyte":"AH cardiomyo.", "AdultLoopOfHenle":"AK loop of henle", "FetalBrainNeurGlioblast_CB_VZ":"FCB glioblast",\
         "AdultProximalTubule":"AK prox. tub.", "FetalLeydigMale":"FG leydig", "SST":"AC SST inh neur.", "KosoyRoussosControlMicroglia":"AC microglia",\
         "FetalBrainFloorPlate":"FB fl. plate", "FetalArterialECs":"FH endoth.", "ASCT":"AC astro.", "FetalBrainCOP":"FB COP",\
         "AMY":"AA neur.", "PVALB":"AC PVALB inh neur.", "ITL23":"AC L2-3 IT neur.", "FetalBrainNeurCB_GNP_IPC_1":"FB inter. prog.", "FetalBrainNeurDAergic":"FB DA neur.",\
          "OGC":"AC Oligo.", "D1Pu":"AP D1 inh neur.", "FetalBrainNeurSerotonergic":"FB 5-HT neur.", "FetalBrainNeurDRG_2":"FS DRG neur.",\
          "FetalHeartPericytes":"FH peri.", "FetalHeartEndocardium":"FH endocard.", "FetalHeartCardiacFibroblasts":"FH fibro.", "FetalBrainNeurPurkinje_6":"FCB Purk. inh neur.",\
          "AdultHeartSmoothMuscle":"AH smooth musc.", "FetalBrainRoofPlate":"FB ro. plate"}

gobp = pd.read_csv("GOBP_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("HPO_AccelEvol_Input_FiltForAccelEvol.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")
    

In [ ]:
from scipy.stats import binomtest
v = pd.read_csv("Supplemental_Tables/Supplemental_Table9.csv")
v.columns = ["Position, hg38", "PhyloP", "Number of species in alignment", "Gene symbol", "Distance to TSS of nearest gene"] + list(v.columns)[5:]
for i in list(v.columns[5:]):
    if i.endswith("l2fc"):
        v[i] = -np.round(v[i], 4)
    elif i.endswith("tile"):
        v[i] = np.round(v[i], 4)*100
v = v.sort_values("PhyloP", ascending = False)
v.to_csv("Supplemental_Tables/Supplemental_Table9.csv")

In [ ]:
def filter_nc(v):
    v["NearestGene"] = v.index
    #Genes that are directly adjacent to centromeres
    blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
                "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
                "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]

    #Genes that are near or in IGV etc. areas
    blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP", "SULT1C3"]
    
    #Genes that seem a little iffy since they are embedded in regions with lots of the same gene
    iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
    orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
    orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
    v = v[v["NearestGene"].isin(orth["Gene name"])]
    v = v[~v["NearestGene"].isin(blacklist + blacklist2)]
    
    olfs = []
    hlas = []
    ugt = []
    for i in v["NearestGene"]:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)

    #Remove olfactory receptors and HLA genes
    v = v[~v["NearestGene"].isin(olfs + hlas + ugt)]

    #Remove the ML-specific blacklisted genes
    bad_ml = list(pd.read_csv("Blacklist_Chimp_NonCanonicalChrom.txt", sep = "\t", header = None)[0])
    v = v[~v["NearestGene"].isin(bad_ml)]

    v = v.drop("NearestGene", axis = 1)
    return v

In [ ]:
#Run for both FiltWGS and NoFiltWGS, need to rerun all this because didn't name columns properly
#Also need to restrict to only genes with at least 50 variants in 

folder = "ML_Results_Download"
c = 0
t = "AbsLogfc_NC_FiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_HPO_Results"
for file in os.listdir(folder):
    if t in file:
        if "GOBP" not in t and "HPO" not in t:
            v = pd.read_csv(folder + "/" + file, sep = "\t").set_index("NearestGene")
        else:
            v = pd.read_csv(folder + "/" + file, sep = "\t").set_index("Term")
        v2 = v[["Z-score Difference"]]
        v2.columns = [file.split("_AbsLogfc")[0]]
        v3 = v[["Z-score Difference Corr Tot"]]
        v3.columns = [file.split("_AbsLogfc")[0]]
        v4 = v[["Z-score Difference Corr S"]]
        v4.columns = [file.split("_AbsLogfc")[0]]
        v2x = v[["Species1 Sum Total_Vars", "Species2 Sum Total_Vars"]]
        v2x.columns = [file.split("_AbsLogfc")[0] + " Human Num Vars", file.split("_AbsLogfc")[0] + " Chimp Num Vars"]
        if not c:
            df2 = v2.copy()
            df3 = v3.copy()
            df4 = v4.copy()
            dfx = v2x.copy()
        else:
            df2 = df2.join(v2)
            df3 = df3.join(v3)
            df4 = df4.join(v4)
            dfx = dfx.join(v2x)
        c += 1
df3 = df3.dropna()
df2 = df2.dropna()
df4 = df4.dropna()
df2 = df2.loc[np.intersect1d(df3.index, df2.index)]
df3 = df3.loc[np.intersect1d(df3.index, df2.index)]
df4 = df4.loc[np.intersect1d(df4.index, df2.index)]

if "GOBP" not in t and "HPO" not in t:
    df4 = filter_nc(df4)
    df3 = filter_nc(df3)
    df2 = filter_nc(df2)

dfx["Max"] = np.max(dfx, axis = 1)
dfx = dfx[dfx["Max"] > 50]
df4 = df4.loc[np.intersect1d(dfx.index, df4.index)]
df3 = df3.loc[np.intersect1d(dfx.index, df3.index)]
df2 = df2.loc[np.intersect1d(dfx.index, df2.index)]


# Apply mapping


df4.to_csv("MASH_ZScore_CorrS_" + t + ".txt", sep = "\t")
df3.to_csv("MASH_ZScore_CorrTot_" + t + ".txt", sep = "\t")
df2.to_csv("MASH_ZScore_NoCorr_" + t + ".txt", sep = "\t")
df4.loc[:, :] = 1
df3.loc[:, :] = 1
df2.loc[:, :] = 1
df4.to_csv("MASH_STDE_CorrS_" + t + ".txt", sep = "\t")
df3.to_csv("MASH_STDE_CorrTot_" + t + ".txt", sep = "\t")
df2.to_csv("MASH_STDE_NoCorr_" + t + ".txt", sep = "\t")


In [ ]:
#Run MASH on all 4 combinations of CorrTot/NoCorr and NoFiltWGS/FiltWGS, read in resulting LFSR
#Use mash_prefilt.R
t = "AbsLogfc_NC_FiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results"

ncnf = pd.read_csv("Lfsr_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
ncf = pd.read_csv("Lfsr_NoCorr_AbsLogfc_NC_FiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")


In [ ]:
# Boolean conditions
condition1_nf = (ncnf < 0.05)
condition2_nf = (ncnf > 0.05)

# Combine conditions and convert to 1/0
result_nf = (condition1_nf | condition2_nf).astype(int)


# Boolean conditions
condition1_f = (ncf < 0.05)
condition2_f = (ncf > 0.05)

# Combine conditions and convert to 1/0
result_f = (condition1_f | condition2_f).astype(int)
result_nf["Sum"] = np.sum(result_nf, axis = 1)
result_f["Sum"] = np.sum(result_f, axis = 1)

keep = np.intersect1d(result_f[result_f["Sum"] >= 25].index, result_nf[result_nf["Sum"] >= 25].index)
len(keep)

In [ ]:

v = pd.read_csv("Effects_Filt_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")



In [ ]:
os.mkdir("KEGG_Relative_Enrichment")

In [ ]:
ncnf = pd.read_csv("Lfsr_Filt_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
ctnf = pd.read_csv("Lfsr_Filt_CorrTot_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
ncf = pd.read_csv("Lfsr_Filt_NoCorr_AbsLogfc_NC_FiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
ctf = pd.read_csv("Lfsr_Filt_CorrTot_AbsLogfc_NC_FiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")

# Boolean conditions
condition1_nf = (ncnf < 0.05) & (ctnf < 0.25)
condition2_nf = (ncnf > 0.05) & (ctnf > 0.05)

# Combine conditions and convert to 1/0
result_nf = (condition1_nf | condition2_nf).astype(int)


# Boolean conditions
condition1_f = (ncf < 0.05) & (ctf < 0.25)
condition2_f = (ncf > 0.05) & (ctf > 0.05)

# Combine conditions and convert to 1/0
result_f = (condition1_f | condition2_f).astype(int)
result_nf["Sum"] = np.sum(result_nf, axis = 1)
result_f["Sum"] = np.sum(result_f, axis = 1)
condition1_nf["Sum"] = np.sum(condition1_nf, axis = 1)
condition1_f["Sum"] = np.sum(condition1_f, axis = 1)
condition2_nf["Sum"] = np.sum(condition2_nf, axis = 1)
condition2_f["Sum"] = np.sum(condition2_f, axis = 1)

keep = np.intersect1d(result_f[result_f["Sum"] >= 25].index, result_nf[result_nf["Sum"] >= 25].index)
keep_sig = np.intersect1d(keep, np.intersect1d(condition1_f[condition1_f["Sum"] >= 1].index, condition1_nf[condition1_nf["Sum"] >= 1].index))
keep_nsig_fpc = np.intersect1d(keep, np.intersect1d(condition2_f[condition2_f["FetalBrainNeurPurkinje_6"] >= 1].index, condition2_nf[condition2_nf["FetalBrainNeurPurkinje_6"] >= 1].index))
print(len(keep), len(keep_sig), len(keep_nsig_fpc))

df = pd.read_csv("Effects_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
lfsr = pd.read_csv("Lfsr_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")

signed_lfsr = np.sign(df.sort_index())*-np.log10(lfsr.sort_index())
signed_lfsr["Mean"] = np.mean(signed_lfsr, axis = 1)

signed_lfsr["Max"] = np.max(signed_lfsr, axis = 1)
signed_lfsr["Min"] = np.min(signed_lfsr, axis = 1)

sig_pos_select = signed_lfsr[(signed_lfsr["Max"] > -np.log10(0.05)) | (signed_lfsr["Min"] < np.log10(0.05))].sort_values("Max")
sig_pos_select["Variance"] = np.var(sig_pos_select.drop(["Mean", "Max", "Min"], axis = 1), axis = 1)/np.mean(sig_pos_select.drop(["Mean", "Max", "Min"], axis = 1), axis = 1)
sig_pos_select = sig_pos_select.loc[np.intersect1d(keep_sig, sig_pos_select.index)].copy().sort_values("Max")
sig_pos_select[["LiangSteinNeuron"]].sort_values("LiangSteinNeuron")

In [ ]:
s = signed_lfsr.loc[np.intersect1d(keep, signed_lfsr.index)].copy().sort_values("Max")
k1 = []
k2 = []
for index, row in s.iterrows():
    if row["Min"] < np.log10(0.05):
        k1.append("Chimpanzee-accelerated CA in at least one cell type")
    else:
        k1.append("Not chimpanzee-accelerated in any cell type")
    if row["Max"] > -np.log10(0.05):
        k2.append("Human-accelerated CA in at least one cell type")
    else:
        k2.append("Not human-accelerated in any cell type")
s["Human-acceleration for CA"] = k2
s["Chimp-acceleration for CA"] = k1
s = s.drop(["Mean"], axis = 1)
n_cols = []
for i in s.columns:
    if i in list(d_abrev.keys()):
        n_cols.append(d_abrev[i])
    else:
        if i == "Max":
            n_cols.append("Maximum signed LFSR across cell types")
        elif i == "Min":
            n_cols.append("Minimum signed LFSR across cell types")
        else:
           n_cols.append(i)
s.columns = n_cols
s.sort_values("Maximum signed LFSR across cell types", ascending = False).to_csv("Supplemental_Tables/Supplemental_Table8.csv")

In [ ]:
signed_lfsr = signed_lfsr.sort_values("Max", ascending = False)
signed_lfsr

In [ ]:
from scipy.stats import binomtest
binomtest(sig_pos_select[sig_pos_select["Max"] > -np.log10(0.05)].shape[0], sig_pos_select[sig_pos_select["Min"] < np.log10(0.05)].shape[0] + sig_pos_select[sig_pos_select["Max"] > -np.log10(0.05)].shape[0])

In [ ]:
plt.figure(figsize=(3, 4), dpi=600)
sns.barplot(x = ["Human", "Chimp"], y = [142, 285-142], hue = ["Human", "Chimp"], palette = palette, alpha = 0.8)
plt.ylabel("Number of genes with\naccelerated CA evolution", size = 12)
plt.ylim(0, 170)
plt.xticks(size = 12)
plt.yticks(size = 12)
plt.title("No more CA acceleration\nin human than chimp", size = 12)
plt.legend([], [], frameon = False)
x1, x2 = 0, 1                 # bar positions
y, h = 150, 1.3             # height of line + vertical offset
plt.plot([x1, x1, x2, x2],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x2) * 0.5, y + h,
         "NS",
         ha='center', va='bottom', fontsize=14)
plt.show()

In [ ]:
#CDH12 is best candidate for cell type-specificity, CSMD1 and ADCYAP1 for non cell type-specificity
#DPP10 also works well for cell type-specificity, specifically in Purkinje cells where it would likely affect electrophyisology
to_plot = "DLGAP2"

sig = pd.DataFrame(signed_lfsr.loc[to_plot])

colors = []
for index, row in sig.iterrows():
    if row[0] > -np.log10(0.05):
        colors.append(palette["Human"])
    elif row[0] < np.log10(0.05):
        colors.append(palette["Chimp"])
    else:
        colors.append("grey")
    if len(colors) == 34:
        break

psd = pd.read_csv("Psd_Filt_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
eff = pd.read_csv("Effects_Filt_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")

categories = [d_abrev[x.replace("_Alu_HPO.txt", "")] for x in list(eff.columns)]
values = list(eff.loc[to_plot])
std_dev = [x for x in list(psd.loc[to_plot])]
# Create the bar plot
fig, ax = plt.subplots(figsize = (14.5, 4), dpi = 600)
bars = ax.bar(categories, values, yerr=std_dev, capsize=3, color=colors, alpha=0.9)

# Add labels and title
ax.set_ylabel('MASHR effect size', size = 16)
ax.set_title('$\it{' + to_plot + '}$', size = 18)
ax.set_xticks(np.arange(len(categories)))
ax.set_xticklabels(categories, rotation = 90, size = 13)
ax.set_xlabel("Cell type", size = 16)
ax.set_xlim([-0.7, 33.7])
# Show the plot
plt.show()

In [ ]:
from adjustText import adjust_text


def mvp(v, xlab, title, stat = 'Abs Logfc Difference', fdr_col = "FDR Difference", x_col = 'Abs Logfc Difference', genes_to_label = None, gene_col = "NearestGene"):
    plt.figure(figsize=(6, 4), dpi=600)
    sig = []
    #Also filtered out sites that were variable in apes or variable across human populations
    for index, row in v.iterrows():
        if row[fdr_col] < 0.05 and row[stat] < 0:
            sig.append("Chimp accelerated")
        elif row[fdr_col] < 0.05 and row[stat] > 0:
            sig.append("Human accelerated")
        else:
            sig.append("Not significant")
    v["Category"] = sig
    v["-Log$_{10}$(FDR)"] = -np.log10(v[fdr_col])
    v = v.sort_values("Category")
    ax = sns.scatterplot(data = v, x = stat, y = "-Log$_{10}$(FDR)", hue = "Category", palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey"})

    xlim1 = min(v[stat]) - abs(floor(0.1*min(v[stat])))
    xlim2 = max(v[stat]) + abs(floor(0.1*max(v[stat])))
    plt.xlim(xlim1, xlim2)
    plt.hlines(-np.log10(0.05), xlim1, xlim2, linestyles="dashed", colors="k")

    # ---------- LABEL GENES ----------
    v["NearestGene"] = v.index
    texts = []
    x_offset = 0.002 * (plt.xlim()[1] - plt.xlim()[0])
    y_offset = 0.003 * (plt.ylim()[1] - plt.ylim()[0])
    if genes_to_label is not None:
        subset = v[v[gene_col].isin(genes_to_label)]

        for _, row in subset.iterrows():
            dx = x_offset if row[stat] > 0 else -x_offset
            dy = y_offset
            texts.append(
                plt.text(
                    row[stat] + dx,
                    row["-Log$_{10}$(FDR)"] + dy,
                    row[gene_col],
                    fontsize=9,
                )
            )

        adjust_text(
            texts,
            ax=ax,
            arrowprops=dict(arrowstyle="-", lw=0.6, color="black"), expand_axes=False
        )
    from matplotlib.patches import FancyArrowPatch

    # Remove any existing arrows (from previous adjustText runs)
    for artist in ax.get_children():
        if isinstance(artist, FancyArrowPatch):
            artist.remove()
    plt.legend(bbox_to_anchor = (1, 1))
    plt.xlabel(xlab, size = 13)
    plt.ylabel("-Log$_{10}$(LFSR)", size = 13)
    plt.xticks(size = 12)
    plt.yticks(size = 12)
    plt.title(title, size = 14)
    plt.legend([], [], frameon = False)
    plt.show()


In [ ]:
lfsr = pd.read_csv("Lfsr_Filt_NoCorr_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
lfsr = lfsr.loc[keep].copy()
v = pd.read_csv("ML_Results_Download/FetalBrainNeurPurkinje_6_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", sep = "\t")
v = v[v["NearestGene"].isin(lfsr.index)].set_index("NearestGene")
v = v.join(lfsr[["FetalBrainNeurPurkinje_6"]])
v = v.loc[list(keep_sig) + list(keep_nsig_fpc)]
v.sort_values("p-value Difference Corr Tot")

mvp(v, "Difference in predicted |log$_{2}$ fold-change in CA|", "Accelerated evolution of CA in $\it{cis}$-regulatory landscapes", stat = 'Abs Logfc Difference', fdr_col = "FetalBrainNeurPurkinje_6", x_col = 'Abs Logfc Difference', genes_to_label = ["CSMD1", "DPP10", "NXN", "TGIF1", "EMILIN2", "CLU", "PSD4"])